# LoRA Fine-Tune MegaDescriptor (Colab + Drive)

Fine-tune **MegaDescriptor-L-384** with **LoRA + ArcFace** on Amvrakikos + Reunion, then evaluate opposite-side re-ID on held-out **Zakynthos**.

## Before you run
1. Open this notebook in **Google Colab**.
2. Set **Runtime → Change runtime type → GPU** (T4 or better).
3. Put datasets under Drive:
   ```
   MyDrive/SeaTurtle/
     AmvrakikosTurtles/
     ReunionTurtles/
     ZakynthosTurtles/
   ```
4. Run cells top to bottom. Cell 1 always pulls the latest repo commit and clears cached imports.
5. Checkpoints land in `MyDrive/SeaTurtle/checkpoints/lora_megadescriptor/` (adapter + ArcFace head + config). Features and eval CSV go to Drive too.

In [5]:
import os
import shutil
import sys
import subprocess

REPO_URL = 'https://github.com/abui-am/side-matching.git'
CLONE_DIR = '/content/sides-matching'

def assert_colab_gpu():
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError(
            'This notebook is Colab-only. Open it in Google Colab with a GPU runtime.'
        ) from exc
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            'CUDA GPU is required. In Colab choose Runtime > Change runtime type > GPU.'
        )
    print(f'Colab GPU: {torch.cuda.get_device_name(0)}')
    return torch

def find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.isdir(os.path.join(path, 'sides_matching')):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    for candidate in [CLONE_DIR, '/content']:
        if os.path.isdir(os.path.join(candidate, 'sides_matching')):
            return candidate
    return None

def _purge_sides_matching_modules():
    for name in list(sys.modules):
        if name == 'sides_matching' or name.startswith('sides_matching.'):
            del sys.modules[name]

def ensure_repo_root():
    """Clone or hard-reset to origin/main so Colab never keeps a stale checkout."""
    root = find_repo_root()
    if root is None:
        print(f'Cloning {REPO_URL} -> {CLONE_DIR}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, CLONE_DIR])
        root = CLONE_DIR
    elif os.path.isdir(os.path.join(root, '.git')):
        print(f'Updating repo at {root} to origin/main...')
        try:
            subprocess.check_call(['git', '-C', root, 'fetch', '--depth', '1', 'origin', 'main'])
            subprocess.check_call(['git', '-C', root, 'reset', '--hard', 'origin/main'])
        except subprocess.CalledProcessError:
            print('git update failed; re-cloning...')
            shutil.rmtree(root)
            subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, CLONE_DIR])
            root = CLONE_DIR
    else:
        print(f'No .git at {root}; re-cloning into {CLONE_DIR}')
        if os.path.isdir(CLONE_DIR):
            shutil.rmtree(CLONE_DIR)
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, CLONE_DIR])
        root = CLONE_DIR
    _purge_sides_matching_modules()
    commit = subprocess.check_output(
        ['git', '-C', root, 'rev-parse', '--short', 'HEAD'], text=True
    ).strip()
    print(f'Repo commit: {commit}')
    return root

def pip_install(*packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *packages]
    print('>', ' '.join(cmd))
    subprocess.check_call(cmd)

def uninstall_incompatible_torchao():
    """Colab ships torchao 0.10; recent peft hard-raises unless torchao is absent or >=0.16.
    We do not use torchao quantization, so uninstalling is safer than upgrading Colab's stack.
    """
    try:
        import importlib.metadata as metadata
        version = metadata.version('torchao')
    except metadata.PackageNotFoundError:
        print('torchao not installed')
        return
    print(f'Removing incompatible torchao {version} (peft requires >=0.16 or absent)')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
        stdout=subprocess.DEVNULL,
    )
    for name in list(sys.modules):
        if name == 'torchao' or name.startswith('torchao.'):
            del sys.modules[name]
    try:
        from peft.import_utils import is_torchao_available
        is_torchao_available.cache_clear()
    except Exception:
        pass

torch = assert_colab_gpu()
repo_root = ensure_repo_root()
if repo_root in sys.path:
    sys.path.remove(repo_root)
sys.path.insert(0, repo_root)

pip_install('wildlife-datasets', 'timm', 'scikit-image', 'peft')
pip_install('git+https://github.com/WildlifeDatasets/wildlife-tools@main')
uninstall_incompatible_torchao()

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/SeaTurtle'
DATASET_DIRS = ('AmvrakikosTurtles', 'ReunionTurtles', 'ZakynthosTurtles')

def has_datasets(path):
    return all(os.path.isdir(os.path.join(path, name)) for name in DATASET_DIRS)

if not has_datasets(DRIVE_DATA):
    raise FileNotFoundError(
        f'Datasets not found at {DRIVE_DATA}. '
        'Put AmvrakikosTurtles, ReunionTurtles, ZakynthosTurtles in Drive > SeaTurtle.'
    )

device = torch.device('cuda')
root_features = os.path.join(DRIVE_DATA, 'features')
checkpoint_dir = os.path.join(DRIVE_DATA, 'checkpoints', 'lora_megadescriptor')
os.makedirs(root_features, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

print(f'Repo: {repo_root}')
print(f'Data: {DRIVE_DATA} OK')
print(f'Features: {root_features}')
print(f'Checkpoints: {checkpoint_dir}')

Colab GPU: Tesla T4
Updating repo at /content/sides-matching to origin/main...
Repo commit: 959e620
> /usr/bin/python3 -m pip install -q wildlife-datasets timm scikit-image peft
> /usr/bin/python3 -m pip install -q git+https://github.com/WildlifeDatasets/wildlife-tools@main
torchao not installed
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo: /content/sides-matching
Data: /content/drive/MyDrive/SeaTurtle OK
Features: /content/drive/MyDrive/SeaTurtle/features
Checkpoints: /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor


In [2]:
import pandas as pd
from sides_matching import amvrakikos, reunion_green, reunion_hawksbill, zakynthos
from sides_matching.train_lora import (
    TrainConfig,
    IdentityMapper,
    merge_train_dataframes,
    split_identities,
    get_train_transform,
    get_eval_transform,
    wildlife_dataset_from_df,
    build_concat_dataloader,
)

config = TrainConfig()
print(
    f'Config: batch_size={config.batch_size}, lora_r={config.lora_r}, '
    f'amp={config.use_amp}, grad_checkpointing={config.grad_checkpointing}'
)
train_transform = get_train_transform(config.img_size)
val_transform = get_eval_transform(flip=False, img_size=config.img_size)

root_data = DRIVE_DATA
train_sources = [
    ('Amvrakikos', os.path.join(root_data, 'AmvrakikosTurtles'), amvrakikos),
    ('ReunionGreen', os.path.join(root_data, 'ReunionTurtles'), reunion_green),
    ('ReunionHawksbill', os.path.join(root_data, 'ReunionTurtles'), reunion_hawksbill),
]

train_frames = []
for name, root, dataset_fn in train_sources:
    dataset = dataset_fn(root, transform=None)
    train_frames.append((name, dataset.df.copy()))
    print(f'{name}: {len(dataset.df)} images, {dataset.df["identity"].nunique()} identities')

merged_df = merge_train_dataframes(train_frames)
train_df, val_df = split_identities(
    merged_df,
    val_fraction=config.val_identity_fraction,
    seed=config.seed,
)
identity_mapper = IdentityMapper.from_identities(merged_df['global_identity'])

print(f'Train images: {len(train_df)} | Val images: {len(val_df)} | Classes: {identity_mapper.num_classes}')

train_datasets = []
train_label_parts = []
val_datasets = []
val_label_parts = []

for name, root, dataset_fn in train_sources:
    source_train = train_df[train_df['source_dataset'] == name].reset_index(drop=True)
    source_val = val_df[val_df['source_dataset'] == name].reset_index(drop=True)
    if len(source_train):
        train_datasets.append(wildlife_dataset_from_df(root, source_train, dataset_fn, train_transform))
        train_label_parts.append(identity_mapper.encode_series(source_train['global_identity']))
    if len(source_val):
        val_datasets.append(wildlife_dataset_from_df(root, source_val, dataset_fn, val_transform))
        val_label_parts.append(identity_mapper.encode_series(source_val['global_identity']))

train_loader = build_concat_dataloader(
    train_datasets, train_label_parts, config.batch_size, shuffle=True,
    num_workers=config.num_workers,
)
val_loader = build_concat_dataloader(
    val_datasets, val_label_parts, config.batch_size, shuffle=False,
    num_workers=config.num_workers,
)

zakynthos_root = os.path.join(root_data, 'ZakynthosTurtles')
zakynthos_dataset = zakynthos(zakynthos_root, transform=None)
print(f'Zakynthos (test only): {len(zakynthos_dataset.df)} images, {zakynthos_dataset.df["identity"].nunique()} identities')

Config: batch_size=2, lora_r=8, amp=True, grad_checkpointing=True
Amvrakikos: 200 images, 50 identities
ReunionGreen: 200 images, 50 identities
ReunionHawksbill: 136 images, 34 identities
Train images: 428 | Val images: 108 | Classes: 134
Zakynthos (test only): 160 images, 40 identities


In [6]:
from sides_matching.train_lora import (
    build_training_model,
    configure_optimizer,
    clear_cuda_memory,
)

for _name in ('model', 'optimizer', 'scaler', 'baseline_model', 'lora_model'):
    if _name in globals():
        del globals()[_name]
clear_cuda_memory()

model = build_training_model(identity_mapper.num_classes, config, device)
optimizer = configure_optimizer(model, config)
scaler = torch.cuda.amp.GradScaler(enabled=config.use_amp and device.type == 'cuda')
model.backbone.print_trainable_parameters()
print(
    f'Train memory settings: batch_size={config.batch_size}, '
    f'lora_r={config.lora_r}, amp={config.use_amp}, '
    f'grad_checkpointing={config.grad_checkpointing}'
)
print(f'GPU mem allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Gradient checkpointing: on
trainable params: 2,312,064 || all params: 197,510,580 || trainable%: 1.1706
Train memory settings: batch_size=2, lora_r=8, amp=True, grad_checkpointing=True
GPU mem allocated: 0.81 GB


/tmp/ipykernel_8967/1897912378.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=config.use_amp and device.type == 'cuda')


In [7]:
from sides_matching.train_lora import (
    train_epoch,
    validate_epoch,
    embedding_recall_at_1,
    save_checkpoint,
    clear_cuda_memory,
    is_cuda_oom,
)

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

best_recall = -1.0
patience_counter = 0
history = []

def rebuild_loaders():
    global train_loader, val_loader
    train_loader = build_concat_dataloader(
        train_datasets, train_label_parts, config.batch_size, shuffle=True,
        num_workers=config.num_workers,
    )
    val_loader = build_concat_dataloader(
        val_datasets, val_label_parts, config.batch_size, shuffle=False,
        num_workers=config.num_workers,
    )

def run_train_epoch():
    return train_epoch(
        model, train_loader, optimizer, device,
        scaler=scaler, use_amp=config.use_amp,
    )

for epoch in range(1, config.epochs + 1):
    while True:
        try:
            train_loss = run_train_epoch()
            break
        except Exception as exc:
            if not is_cuda_oom(exc):
                raise
            clear_cuda_memory()
            if config.batch_size <= 1:
                raise RuntimeError(
                    'CUDA OOM even at batch_size=1. Restart runtime and re-run from cell 1.'
                ) from exc
            config.batch_size = max(1, config.batch_size // 2)
            rebuild_loaders()
            print(f'CUDA OOM: reduced batch_size to {config.batch_size}, retrying epoch...')

    val_loss, val_recall = validate_epoch(
        model, val_loader, device, use_amp=config.use_amp,
    )
    embed_recall = embedding_recall_at_1(
        model, val_loader, device, use_amp=config.use_amp,
    )
    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_recall': val_recall,
        'embed_recall': embed_recall,
        'batch_size': config.batch_size,
    })
    print(
        f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | '
        f'val_recall@1={val_recall:.4f} | embed_recall@1={embed_recall:.4f} | '
        f'batch={config.batch_size}'
    )

    if embed_recall > best_recall:
        best_recall = embed_recall
        patience_counter = 0
        save_checkpoint(
            model,
            identity_mapper,
            config,
            checkpoint_dir,
            metrics={'embed_recall_at_1': embed_recall, 'epoch': epoch},
        )
        print(f'Saved best checkpoint to {checkpoint_dir}')
    else:
        patience_counter += 1
        if patience_counter >= config.early_stop_patience:
            print('Early stopping triggered.')
            break

history_df = pd.DataFrame(history)
display(history_df.tail())

Epoch 01 | train_loss=13.1122 | val_loss=16.5183 | val_recall@1=0.0000 | embed_recall@1=0.7407 | batch=2
Saved best checkpoint to /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor
Epoch 02 | train_loss=9.3985 | val_loss=17.5786 | val_recall@1=0.0000 | embed_recall@1=0.7407 | batch=2
Epoch 03 | train_loss=6.5161 | val_loss=17.7374 | val_recall@1=0.0000 | embed_recall@1=0.7407 | batch=2
Epoch 04 | train_loss=4.2432 | val_loss=17.9068 | val_recall@1=0.0000 | embed_recall@1=0.7593 | batch=2
Saved best checkpoint to /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor
Epoch 05 | train_loss=3.1875 | val_loss=18.0841 | val_recall@1=0.0000 | embed_recall@1=0.7778 | batch=2
Saved best checkpoint to /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor
Epoch 06 | train_loss=1.7719 | val_loss=18.4263 | val_recall@1=0.0000 | embed_recall@1=0.7778 | batch=2
Epoch 07 | train_loss=1.5836 | val_loss=18.5001 | val_recall@1=0.0000 | embed_recall@1=0.7870 | batch=2
S

,epoch,train_loss,val_loss,val_recall,embed_recall,batch_size
11,12,0.703999,19.048886,0.0,0.796296,2
12,13,0.429917,19.138893,0.0,0.787037,2
13,14,0.526670,19.230783,0.0,0.787037,2
14,15,0.476486,19.254300,0.0,0.814815,2
15,16,0.437937,19.500050,0.0,0.787037,2


In [8]:
from sides_matching.train_lora import build_inference_model, extract_features, save_feature_pickle, get_eval_transform, clear_cuda_memory
from wildlife_tools.features import DeepFeatures
from sides_matching import get_features, get_transform
import timm

adapter_dir = os.path.join(checkpoint_dir, 'adapter')
if not os.path.isdir(adapter_dir):
    raise FileNotFoundError(f'Missing adapter at {adapter_dir}. Run training first.')

# Free training graph before feature extraction.
for _name in ('model', 'optimizer', 'scaler'):
    if _name in globals():
        del globals()[_name]
clear_cuda_memory()

lora_model = build_inference_model(config, adapter_dir, device)
grayscale = False
extract_batch = max(1, min(config.batch_size, 4))

for flip in [True, False]:
    transform = get_eval_transform(flip=flip, img_size=config.img_size)
    eval_dataset = zakynthos(zakynthos_root, transform=transform)
    features = extract_features(lora_model, eval_dataset, device, batch_size=extract_batch)
    file_name = os.path.join(
        root_features,
        f'MegaDescriptorLoRA_Zakynthos_flip={flip}_grayscale={grayscale}.pickle',
    )
    save_feature_pickle(features, file_name)
    print(f'Saved {file_name} shape={features.shape}')

del lora_model
clear_cuda_memory()

baseline_model = timm.create_model(config.model_name, num_classes=0, pretrained=True).to(device)
baseline_model.eval()
baseline_extractor = DeepFeatures(baseline_model, batch_size=extract_batch, device=device)

for flip in [True, False]:
    transform = get_transform(flip=flip, grayscale=grayscale, img_size=config.img_size, normalize=True)
    eval_dataset = zakynthos(zakynthos_root, transform=transform)
    file_name = os.path.join(
        root_features,
        f'MegaDescriptor_Zakynthos_flip={flip}_grayscale={grayscale}.pickle',
    )
    if not os.path.exists(file_name):
        get_features(file_name, eval_dataset, baseline_extractor)
        print(f'Extracted baseline {file_name}')
    else:
        print(f'Using existing baseline {file_name}')

del baseline_model, baseline_extractor
clear_cuda_memory()
print('Feature extraction done for base + LoRA.')

Saved /content/drive/MyDrive/SeaTurtle/features/MegaDescriptorLoRA_Zakynthos_flip=True_grayscale=False.pickle shape=(160, 1536)
Saved /content/drive/MyDrive/SeaTurtle/features/MegaDescriptorLoRA_Zakynthos_flip=False_grayscale=False.pickle shape=(160, 1536)
Using existing baseline /content/drive/MyDrive/SeaTurtle/features/MegaDescriptor_Zakynthos_flip=True_grayscale=False.pickle
Using existing baseline /content/drive/MyDrive/SeaTurtle/features/MegaDescriptor_Zakynthos_flip=False_grayscale=False.pickle


In [ ]:
from sides_matching.train_lora import compare_base_vs_lora

mods = [
    'full',
    'same orientation',
    'different orientation',
    'same year',
    'different year',
    'different both',
]

results_df, comparison_df = compare_base_vs_lora(
    zakynthos_dataset.df,
    root_features,
    flips=(True, False),
    grayscale=False,
    mods=mods,
)

print('Base MegaDescriptor vs LoRA fine-tune on held-out Zakynthos')
print('delta = LoRA - base (positive means LoRA is better)')
display(comparison_df.round(4))

# Headline split for the paper: opposite-side matching
focus = comparison_df[comparison_df['mod'] == 'different orientation'].copy()
print('\nOpposite-side (different orientation) Top-1')
display(focus[['flip', 'base_top1', 'lora_top1', 'delta_top1']].round(4))

results_csv = os.path.join(checkpoint_dir, 'zakynthos_eval.csv')
comparison_csv = os.path.join(checkpoint_dir, 'zakynthos_base_vs_lora.csv')
results_df.to_csv(results_csv, index=False)
comparison_df.to_csv(comparison_csv, index=False)
print(f'Saved {results_csv}')
print(f'Saved {comparison_csv}')

In [ ]:
# Confirm trained model is on Google Drive (and zip for easy download/share)
import glob

required = [
    os.path.join(checkpoint_dir, 'adapter'),
    os.path.join(checkpoint_dir, 'arcface_head.pt'),
    os.path.join(checkpoint_dir, 'train_config.json'),
]
missing = [path for path in required if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(
        'Training checkpoint incomplete. Missing:\n  - ' + '\n  - '.join(missing)
    )

adapter_files = sorted(glob.glob(os.path.join(checkpoint_dir, 'adapter', '*')))
print(f'Checkpoint OK at {checkpoint_dir}')
for path in required:
    print(f'  - {path}')
print(f'  - adapter files ({len(adapter_files)}):')
for path in adapter_files:
    print(f'      {os.path.basename(path)}')

zip_base = os.path.join(DRIVE_DATA, 'checkpoints', 'lora_megadescriptor_bundle')
shutil.make_archive(zip_base, 'zip', checkpoint_dir)
print(f'\nZipped model -> {zip_base}.zip')
print('Open: drive.google.com → MyDrive → SeaTurtle → checkpoints/')